# 解码策略：从 Logits 到下一个 Token

> 前面的章节已经把 Transformer 的前向计算讲清楚了。模型接收一串 Token ID，经过多层计算，最后不会直接吐出文字，而是输出词表中每个 Token 的一组 **logits**。
>
> 本章只回答一个问题：**有了 logits，模型到底怎样决定下一个 Token？**
>
> 1. **Greedy Decoding**：每次选最大值，为什么稳定，却容易失去其他路径？
> 2. **Temperature**：怎样改变概率分布的“尖锐程度”？
> 3. **Top-k / Top-p**：为什么要先删掉尾部候选再采样？
> 4. **Repetition Penalty / Beam Search / Chat Template**：真实生成系统还会做哪些处理？
>
> 这一章不再训练一个玩具模型。我们要研究的是 **logits 之后发生什么**，直接从一组可控 logits 开始，反而更容易看清每个策略到底改了哪里。

假设模型在当前位置给出 8 个候选：

```text
Token       猫     狗     汽车    的     是     跑     。     <eos>
Logit      4.2    3.8    0.5    1.4   1.1   0.8   1.8   0.2
```

logit 越大，模型越偏向这个 Token，但它还不是概率。第一步先做 Softmax。

In [ ]:
import torch
import torch.nn.functional as F

tokens = ["猫", "狗", "汽车", "的", "是", "跑", "。", "<eos>"]
logits = torch.tensor([4.2, 3.8, 0.5, 1.4, 1.1, 0.8, 1.8, 0.2])
probs = F.softmax(logits, dim=-1)

print(f"{'Token':<8} {'Logit':>8} {'Prob':>10}")
print("-" * 30)
for tok, logit, prob in zip(tokens, logits, probs):
    print(f"{tok:<8} {logit.item():>8.2f} {prob.item():>9.2%}")
print("\n概率之和 =", probs.sum().item())

## 1. Greedy：如果每次都选最大的，会发生什么？

Softmax 以后，`猫` 的概率最高。最直接的方法就是：

```text
找到最大概率
→ 选中对应 Token
→ 把它追加到输入
→ 再跑一次模型
```

这就是 **Greedy Decoding**。

它的优点很直接：实现简单、结果确定、没有采样随机性。相同输入和模型下，每次都会选同一个 Token。

但它也埋下第一个问题：**当前一步最优，不代表整段序列最优。**

In [ ]:
greedy_id = torch.argmax(logits).item()
print("Greedy 选择:", tokens[greedy_id])
print("对应概率:", f"{probs[greedy_id].item():.2%}")

如果模型后续形成两条路径：

```text
猫 → 在 → 沙发上
狗 → 正在 → 草地上跑
```

Greedy 在第一步选了 `猫` 后，`狗` 路径就彻底消失了。

所以继续追问：

> **如果不想永远选第一名，能不能让第二名、第三名也有机会？**

可以。最简单的方式是按概率采样。但采样之前，先学会控制概率分布本身。

## 2. Temperature：不改排名，只改“自信程度”

公式是：

\[
p_i = \operatorname{softmax}(z_i / T)
\]

- `T < 1`：分布更尖，第一名更占优势。
- `T = 1`：保持原始分布。
- `T > 1`：分布更平，低概率 Token 更容易被采到。

关键点：**Temperature 通常不改变排名，只改变候选之间的差距。**

In [ ]:
temperatures = [0.2, 0.7, 1.0, 1.5]

for T in temperatures:
    p = F.softmax(logits / T, dim=-1)
    top = torch.topk(p, 3)
    print(f"\nT={T}")
    for idx, prob in zip(top.indices.tolist(), top.values.tolist()):
        print(f"  {tokens[idx]:<6} {prob:>7.2%}")

观察时只看一件事：

```text
低温：第一名几乎垄断
高温：第一名优势缩小，候选变多
```

但 Temperature 又带来新问题。

真实词表可能有十几万个 Token。即使尾部 Token 概率很低，它们仍然不是 0。温度升高后，尾部概率也会一起被抬高。

所以真实系统通常不会直接在整个词表里采样，而是先做候选截断。

## 3. Top-k：只留下前 k 个候选

Top-k 的规则很机械：

```text
按概率排序
→ 只保留最高的 k 个
→ 其他 logits 设成 -inf
→ 重新 Softmax
→ 再采样
```

如果 `k=3`，当前例子只剩 `猫 / 狗 / 。` 三个候选。

In [ ]:
def top_k_filter(logits, k):
    if k is None or k >= logits.numel():
        return logits.clone()
    values, _ = torch.topk(logits, k)
    threshold = values[-1]
    out = logits.clone()
    out[out < threshold] = float("-inf")
    return out

filtered = top_k_filter(logits, k=3)
p = F.softmax(filtered, dim=-1)

print("Top-k=3 之后：")
for tok, prob in zip(tokens, p):
    if prob.item() > 0:
        print(f"  {tok:<6} {prob.item():.2%}")

Top-k 解决了“候选太多”，但它也有缺陷：

> `k` 是固定的。

如果模型非常确定，保留几十个候选没有必要；如果模型面对开放问题，合理候选很多，固定的 `k` 又可能截得太狠。

我们希望候选数量能随着概率分布自动变化。

## 4. Top-p：让候选数量跟着分布变化

Top-p 也叫 **Nucleus Sampling**。

它不规定“留几个 Token”，而是规定：

> 从最高概率开始累加，直到累计概率超过 `p`。

```text
猫  0.48  → 累计 0.48
狗  0.32  → 累计 0.80
。  0.10  → 累计 0.90
```

如果 `p=0.8`，候选可能只有 `猫` 和 `狗`。如果分布很平，达到 0.8 可能需要很多 Token。

In [ ]:
def top_p_filter(logits, p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    remove = cumulative > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    sorted_logits = sorted_logits.clone()
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out[sorted_idx] = sorted_logits
    return out

for p_value in [0.5, 0.8, 0.95]:
    filtered = top_p_filter(logits, p_value)
    probs_p = F.softmax(filtered, dim=-1)
    kept = [tokens[i] for i in range(len(tokens)) if probs_p[i] > 0]
    print(f"p={p_value}: 保留 {len(kept)} 个 -> {kept}")

到这里，最常见的采样链已经出现：

```text
logits
↓ logits processor / repetition penalty
temperature
↓
top-k / top-p
↓
softmax
↓
multinomial sample
↓
next token
```

以后在框架源码里看到这些词，先问：

> **这个操作是在改 logits、删候选，还是从概率里选 Token？**

In [ ]:
def sample_next_token(logits, temperature=1.0, top_k=None, top_p=None, seed=42):
    torch.manual_seed(seed)
    x = logits.clone() / max(temperature, 1e-5)
    if top_k is not None:
        x = top_k_filter(x, top_k)
    if top_p is not None:
        x = top_p_filter(x, top_p)
    probs = F.softmax(x, dim=-1)
    token_id = torch.multinomial(probs, 1).item()
    return token_id, probs

configs = [
    ("低温", dict(temperature=0.3)),
    ("高温", dict(temperature=1.5)),
    ("Top-k", dict(temperature=1.0, top_k=3)),
    ("Top-p", dict(temperature=1.0, top_p=0.8)),
]
for name, cfg in configs:
    token_id, _ = sample_next_token(logits, **cfg)
    print(f"{name:<8} -> {tokens[token_id]}")

## 5. Repetition Penalty：为什么模型会“复读”？

自回归生成常见一个现象：已经出现过的 Token，因为概率仍然很高，被反复选中，最后陷入循环。

一种简单处理是 **Repetition Penalty**：对已经出现过的 Token 调低 logit。

In [ ]:
def apply_repetition_penalty(logits, seen_ids, penalty=1.2):
    out = logits.clone()
    for idx in set(seen_ids):
        score = out[idx]
        out[idx] = score / penalty if score > 0 else score * penalty
    return out

seen = [0, 3]
penalized = apply_repetition_penalty(logits, seen, penalty=1.3)
print("原始 P(猫):", f"{F.softmax(logits, -1)[0].item():.2%}")
print("惩罚后 P(猫):", f"{F.softmax(penalized, -1)[0].item():.2%}")

## 6. Beam Search：为什么“每一步第一名”不等于“整句第一名”？

Greedy 和 Sampling 最终都只保留一条序列。

Beam Search 会同时保留 `B` 条路径：

```text
第 1 步保留 B 条路径
↓
每条路径继续展开候选
↓
比较累计 log probability
↓
继续保留最好的 B 条
```

它常见于翻译、语音识别等更强调全局序列得分的任务。开放式 Chat LLM 中，Sampling 通常更常见。

In [ ]:
paths = {
    "A": {"first": 0.60, "second": {"x": 0.40, "y": 0.30}},
    "B": {"first": 0.40, "second": {"x": 0.95, "y": 0.02}},
}
greedy_score = paths["A"]["first"] * max(paths["A"]["second"].values())
beam_best_score = max(paths[p]["first"] * prob for p in paths for prob in paths[p]["second"].values())
print("Greedy 路径总概率:", round(greedy_score, 3))
print("全局最佳路径总概率:", round(beam_best_score, 3))

## 7. Chat Template：模型收到的不是 `messages` JSON

OpenAI-compatible API 里常传：

```json
[
  {"role": "system", "content": "..."},
  {"role": "user", "content": "..."}
]
```

但模型本身仍然只接收 Token 序列。

**Chat Template** 会把多轮角色消息拼成模型训练时熟悉的文本格式，再 Tokenize。

所以部署模型时，如果 Chat Template 错了，会出现一个典型问题：

> 服务能跑，模型也能出字，但回答风格和质量明显不对。

后面的 vLLM / SGLang 部署章会再次碰到它。

## 8. 把这一章放回完整推理链

```text
Prompt / 已生成 Token
        ↓
Transformer Forward
        ↓
      logits
        ↓
Sampling / Decoding
        ↓
   next token
```

本章只处理了下面这一半：**logits → next token**。

下一章继续追问：

> 每生成一个 Token 都要跑一次 Transformer Forward。  
> 如果要生成 1000 个 Token，为什么这件事会这么慢？

这会把我们带到 **Prefill、Decode、KV Cache、MHA / GQA / MQA，以及 TTFT / TPOT**。

## 小结

- **Greedy**：每一步取最大概率。
- **Temperature**：改变概率分布的尖锐程度。
- **Top-k**：固定保留前 k 个候选。
- **Top-p / Nucleus Sampling**：按累计概率动态保留候选。
- **Repetition Penalty**：压低已经出现过的 Token。
- **Beam Search**：同时保留多条候选序列。
- **Chat Template**：把角色消息转换成模型真正接收的 Token 序列。

## 作业

1. 给定 logits `[3.0, 2.0, 1.0]`，分别计算 `T=0.5` 和 `T=2.0` 后的 Softmax。
2. 自己实现 `top_k_filter`，要求 `k=1` 时等价于 Greedy。
3. 给定概率 `[0.4, 0.3, 0.15, 0.1, 0.05]`，手算 `top_p=0.8` 时保留几个 Token。
4. 思考：为什么 Chat LLM 更常用 sampling，而不是 Beam Search？